### Import Libraries

In [ ]:
import os
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

### Import data

In [2]:
image_paths = []
labels = []
classes = 43
base_path = Path("data_gtsrb/Train")

for i in range(classes):
    path = base_path / str(i)
    images = os.listdir(path) 

    for j in images:
        try:
            image_file_path = path / j
            image_paths.append(str(image_file_path))
            labels.append(i)
        except:
            pass


train_paths, test_paths, y_train, y_test = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels)

### Dataset and dataloader

In [3]:
class GTSRBDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

In [4]:
train_dataset = GTSRBDataset(train_paths, y_train, transform=data_transforms)
test_dataset = GTSRBDataset(test_paths, y_test, transform=data_transforms)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

### Model

In [5]:
class MobileNetV2TrafficSigns(nn.Module):
    def __init__(self, num_classes=43):
        super(MobileNetV2TrafficSigns, self).__init__()

        weights = MobileNet_V2_Weights.DEFAULT
        self.base_model = mobilenet_v2(weights=weights)

        for param in self.base_model.parameters():
            param.requires_grad = False # freeze weights

        # allow to change only the final layer
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        x = self.base_model(x)
        return x

In [6]:
model = MobileNetV2TrafficSigns(num_classes=43)

print(model.base_model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=43, bias=True)
)


### Train model

In [15]:
from tqdm import tqdm
import sys

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

epoch = 5
print(f"Training is run on: {device}")

for i in range(epoch):
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Epoch [{i+1}/{epoch}]", leave=True, file=sys.stdout)
    
    for image, label in loop:
        image = image.to(device)
        label = label.to(device)
        
        optimizer.zero_grad()
        pred = model(image)
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        loop.set_postfix(loss=loss.item())
        
    print(f'Epoch [{i+1}/{epoch}] | Training loss: {running_loss/len(train_loader):.4f}')

Training is run on: cpu
Epoch [1/5]: 100%|██████████| 123/123 [13:01<00:00,  6.35s/it, loss=1.49]
Epoch [1/5] | Training loss: 2.1829
Epoch [2/5]: 100%|██████████| 123/123 [16:14<00:00,  7.92s/it, loss=1.08]
Epoch [2/5] | Training loss: 1.2573
Epoch [3/5]: 100%|██████████| 123/123 [30:17<00:00, 14.78s/it, loss=0.878] 
Epoch [3/5] | Training loss: 0.9835
Epoch [4/5]: 100%|██████████| 123/123 [13:35<00:00,  6.63s/it, loss=0.769]
Epoch [4/5] | Training loss: 0.8317
Epoch [5/5]: 100%|██████████| 123/123 [31:41<00:00, 15.46s/it, loss=0.795]  
Epoch [5/5] | Training loss: 0.7367


#### Save the model

In [18]:
torch.save(model.state_dict(), "our_model.pth")

#### Test the model

In [8]:
model = MobileNetV2TrafficSigns()
model.load_state_dict(torch.load("our_model.pth", map_location=torch.device('cpu')))
model.eval()


MobileNetV2TrafficSigns(
  (base_model): MobileNetV2(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
total = 0
correct = 0
running_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images) 
        loss = criterion(outputs, labels) # average loss for a single image from a batch
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        running_loss += loss.item() * images.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
test_loss = running_loss / total

print(f'\nTest accuracy: {test_accuracy}, Test loss: {test_loss}')


Test accuracy: 81.08900790614639, Test loss: 0.7170738199936192


#### Exporting to ONNX to later use in C++

In [18]:
# dummy input tensor matching the expected model dimensions
input = torch.randn(1, 3, 224, 224) # batch size, colors, height, width

torch.onnx.export(
    model,                    
    input,                
    "traffic_sign_model.onnx", 
    export_params=True, # store the trained parameter weights
    opset_version=11, # ONNX operator set version for compatibility
    do_constant_folding=True, # optimize the graph by pre-calculating constant operations
    input_names=['input'],   
    output_names=['output'] 
)

W0527 21:16:51.885000 3172 torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2TrafficSigns([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2TrafficSigns([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


c:\Program Files\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\konio\AppData\Roaming\Python\Python313\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Users\konio\AppData\Roaming\Python\Python313\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\konio\AppData\Roaming\Python\Python313\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\konio\AppData\Roaming\Python\Python313\site-packages\onnx\version_converter

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.12.0+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"output"<FLOAT,[1,43]>
            ),
            initializers=(
                %"base_model.features.0.0.weight"<FLOAT,[32,3,3,3]>{Tensor(...)},
                %"base_model.features.1.conv.0.0.weight"<FLOAT,[32,1,3,3]>{Tensor(...)},
                %"base_model.features.1.conv.1.weight"<FLOAT,[16,32,1,1]>{Tensor(...)},
                %"base_model.features.2.conv.1.0.weight"<FLOAT,[96,1,3,3]>{Tensor(...)},
                %"base_model.classifier.1.bias"<FLOAT,[43]>{TorchTensor(...)},
                %"base_model.features.2.conv.0.0.weight"<FLOAT,[96,16,1,1]>{Tensor(...)},
         

In [ ]:
torch.onnx.checker.check_model()